In [2]:
import numpy as np
import os

# Load the .npy files
# X_train = np.load( "task1_X_train.npy")
y_train = np.load( "/home/sombit-ng-nonadm/vaibhav_intern_pg/BTP+AIproject/task2_damage_state_1/task2_y_train.npy")
# X_test = np.load( "/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1/task2_X_test.npy")
# y_test = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1/task2_y_test.npy")

# Print shapes of the arrays
# print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
# print(f"X_test shape: {X_test.shape}")
# print(f"y_test shape: {y_test.shape}")


y_train shape: (11811, 2)


In [3]:
import numpy as np

file_path = "/home/sombit-ng-nonadm/vaibhav_intern_pg/BTP+AIproject/task2_damage_state_1/task2_X_train.npy"

X_train = np.load(file_path, allow_pickle=True)  # Allow pickle loading
print(f"X_train shape: {X_train.shape}")




X_train shape: (11811, 224, 224, 3)


In [10]:
import os
import cv2
import numpy as np

output_folder = "X_test_images"
os.makedirs(output_folder, exist_ok=True)

for i, img_array in enumerate(X_test):
    image_path = os.path.join(output_folder, f"image_{i}.jpg")
    
    # Convert from float (0-1) or normalized range to uint8 (0-255)
    img_array = (img_array * 255).clip(0, 255).astype(np.uint8)
    
    # Save image
    cv2.imwrite(image_path, img_array)

print(f"Saved {len(X_test)} images to {output_folder}/")


Saved 1460 images to X_test_images/


In [11]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import pandas as pd



In [6]:
from sklearn.model_selection import train_test_split

def split_data(X, y, splitsize=0.2, shuffle=True, stratify=False, seed=42):
    if stratify:
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=splitsize, stratify=y, random_state=seed)
    else:
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=splitsize, random_state=seed)

    return X_train, X_val, y_train, y_val



ProcessData

In [25]:


def image_generators(train_dir, val_dir, X_train_df, X_val_df, batch_size, image_height, image_width):
    """
    Creates image data generators for training and validation.

    Args:
        train_dir (str): Path to training image directory.
        val_dir (str): Path to validation image directory.
        X_train_df (pd.DataFrame): DataFrame with 'image' column (filenames) and 'label' column (classes).
        X_val_df (pd.DataFrame): DataFrame with 'image' column (filenames) and 'label' column (classes).
        batch_size (int): Batch size for training.
        image_height (int): Target image height.
        image_width (int): Target image width.

    Returns:
        train_generator, val_generator
    """

    # Data augmentation for training images
    train_datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        rescale=1./255
    )

    # Only rescale validation images
    val_datagen = ImageDataGenerator(rescale=1./255)

    # Training generator
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=X_train_df,
        directory=train_dir,  # Path to actual image folder
        x_col='image',        # Column with filenames
        y_col='label',        # Column with class labels
        target_size=(image_height, image_width),
        batch_size=batch_size,
        class_mode='binary'  # Use 'categorical' for multi-class classification
    )

    # Validation generator
    val_generator = val_datagen.flow_from_dataframe(
        dataframe=X_val_df,
        directory=val_dir,  # Path to actual image folder
        x_col='image',
        y_col='label',
        target_size=(image_height, image_width),
        batch_size=batch_size,
        class_mode='binary'
    )

    return train_generator, val_generator


In [13]:
X_train, X_val, y_train, y_val= split_data(X_train, y_train, 0.2, True, True, 42)

In [24]:
import os
import cv2
import numpy as np
import pandas as pd

# Define folder to save images
output_dir = "train_images"
os.makedirs(output_dir, exist_ok=True)

# Save images and create file paths
image_paths = []
for i, img_array in enumerate(X_train):
    image_path = os.path.join(output_dir, f"image_{i}.jpg")
    
    # ✅ Convert floating-point values (0-1) to integers (0-255)
    img_array = (img_array * 255).clip(0, 255).astype(np.uint8)  
    
    cv2.imwrite(image_path, img_array)  # Save image
    image_paths.append(image_path)

# Convert labels to class indices
labels = np.argmax(y_train, axis=1)

# Create DataFrame
X_train_df = pd.DataFrame({"image": image_paths, "label": labels})



In [9]:

from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Dropout, GlobalMaxPooling2D, Dense

def build_model(num_classes):
    """
    Loads VGG16 with pre-trained ImageNet weights and modifies the classifier.
    
    Args:
        num_classes (int): Number of output classes.

    Returns:
        model (tensorflow.keras.Model): Modified VGG16 model.
    """

    # Load VGG16 base model (without the fully connected top)
    vgg = VGG16(include_top=False, weights='imagenet', input_shape=(224, 224, 3))

    # Freeze convolutional layers
    for layer in vgg.layers:
        layer.trainable = False

    # Add custom classification head
    x = GlobalMaxPooling2D()(vgg.output)  
    x = Dense(512, activation='relu')(x)  
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)  

    # Create final model
    model = Model(inputs=vgg.input, outputs=x)

    return model

# Example usage
if __name__ == "__main__":
    model = build_model(num_classes=2)  # Example: Binary classification
    model.summary()  # Print model architecture


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 512)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,978,370 (57.14 MB)

 Trainable params: 263,682 (1.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disable GPU

import tensorflow as tf
print("Running on CPU:", tf.config.list_physical_devices('GPU'))


2025-03-19 13:45:57.961987: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-19 13:45:57.975863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742391957.991030  227752 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742391957.995586  227752 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742391958.007542  227752 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Running on CPU: []


2025-03-19 13:46:01.107206: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-03-19 13:46:01.107238: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-03-19 13:46:01.107243: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-03-19 13:46:01.107247: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-03-19 13:46:01.107251: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: hela
2025-03-19 13:46:01.107254: I external/local_xla/xla/stream_executor/cuda/cuda_dia

In [18]:
import numpy as np
import pandas as pd
import os
import cv2

# Define output folder
output_folder = "train_images"
os.makedirs(output_folder, exist_ok=True)

# Save images
X_train_filenames = []
for i, img_array in enumerate(X_train):
    filename = f"image_{i}.jpg"
    filepath = os.path.join(output_folder, filename)
    cv2.imwrite(filepath, img_array * 255)  # Convert from normalized [0,1] to [0,255]
    X_train_filenames.append(filename)

# Convert one-hot encoded y_train (2D) to class labels (1D)
y_train_labels = np.argmax(y_train, axis=1)  # Converts [[1,0], [0,1]] → [0,1]

# Now create DataFrame
X_train_df = pd.DataFrame({'image': X_train_filenames, 'label': y_train_labels})


In [19]:
val_output_folder = "val_images"
os.makedirs(val_output_folder, exist_ok=True)

# Save validation images
X_val_filenames = []
for i, img_array in enumerate(X_val):
    filename = f"val_image_{i}.jpg"
    filepath = os.path.join(val_output_folder, filename)
    cv2.imwrite(filepath, img_array * 255)  # Convert from normalized [0,1] to [0,255]
    X_val_filenames.append(filename)
y_val_labels = np.argmax(y_val, axis=1)

# Create DataFrame for validation dataset
X_val_df = pd.DataFrame({'image': X_val_filenames, 'label': y_val_labels})

In [26]:
path="/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1"
image_size=X_train[0].shape
batch_size = 32
img_height = image_size[1]
img_width = image_size[0]
# X_train_df = pd.DataFrame({'image': X_train, 'label': y_train})
# X_val_df = pd.DataFrame({'image': X_val, 'label': y_val})
# Convert labels (0 → "damaged", 1 → "undamaged")
X_train_df["label"] = X_train_df["label"].astype(str)
X_val_df["label"] = X_val_df["label"].astype(str)

# Example usage
train_generator, val_generator = image_generators(
    train_dir="/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/train_images",   # Folder where training images are stored
    val_dir="/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/val_images",       # Folder where validation images are stored
    X_train_df=X_train_df, 
    X_val_df=X_val_df, 
    batch_size=32, 
    image_height=image_size[1], 
    image_width=image_size[0]
)


Found 7558 validated image filenames belonging to 2 classes.
Found 1890 validated image filenames belonging to 2 classes.


In [23]:
print(X_train_df.head())  # Should show filenames and labels
print(X_val_df.head())    # Should show filenames and labels


         image label
0  image_0.jpg     0
1  image_1.jpg     1
2  image_2.jpg     0
3  image_3.jpg     1
4  image_4.jpg     0
             image label
0  val_image_0.jpg     0
1  val_image_1.jpg     0
2  val_image_2.jpg     1
3  val_image_3.jpg     1
4  val_image_4.jpg     1


In [27]:
from tensorflow.keras.optimizers import Adam

model = build_model(num_classes=2)  # Change num_classes for multi-class

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Train the model
model.fit(train_generator, validation_data=val_generator, epochs=10)


/home/sombit-ng-nonadm/miniconda3/envs/vaibhav_intern/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 2)